# LoRA: Low-Rank Adaptation of Large Language Models

## Learning Objectives

1. Understand low-rank decomposition and why it works for fine-tuning
2. Implement LoRA layers from scratch
3. Fine-tune a model with LoRA using PEFT library
4. Compare LoRA vs. full fine-tuning in terms of memory and speed
5. Merge LoRA adapters for deployment

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(42)
torch.manual_seed(42)

print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Level 1: Understanding Low-Rank Decomposition

Instead of updating full weight matrix W, update low-rank W_delta = B*A

In [ ]:
# Let's understand why low-rank works
# Example: weight matrix for a linear layer

d_in = 768   # Input dimension (e.g., from BERT)
d_out = 3072 # Output dimension (e.g., feed-forward)
r = 8        # Rank (much smaller)

# Full fine-tuning
full_weights = torch.randn(d_out, d_in)
full_params = d_out * d_in

# LoRA: two low-rank matrices
B = torch.randn(d_out, r)  # d_out x r
A = torch.randn(r, d_in)   # r x d_in
lora_params = d_out * r + r * d_in

print(f"Full fine-tuning:")
print(f"  Weight matrix shape: ({d_out}, {d_in})")
print(f"  Parameters: {full_params:,}")
print()
print(f"LoRA fine-tuning (rank={r}):")
print(f"  B matrix shape: ({d_out}, {r})")
print(f"  A matrix shape: ({r}, {d_in})")
print(f"  Total parameters: {lora_params:,}")
print()
print(f"Memory savings: {full_params / lora_params:.1f}x")
print(f"LoRA uses {lora_params / full_params * 100:.1f}% of full parameters")

## Level 2: Implement LoRA Layer from Scratch

A linear layer where W_new = W_old + (B @ A)

In [ ]:
class LoRALinear(nn.Module):
    """Linear layer with LoRA adaptation"""
    
    def __init__(self, in_features, out_features, rank=8, lora_alpha=16, dtype=torch.float32):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.lora_alpha = lora_alpha
        
        # Original weight matrix (frozen)
        self.weight = nn.Parameter(torch.randn(out_features, in_features, dtype=dtype))
        self.bias = nn.Parameter(torch.zeros(out_features, dtype=dtype))
        
        # Freeze original weights
        self.weight.requires_grad = False
        self.bias.requires_grad = False
        
        # LoRA matrices (trainable)
        self.lora_A = nn.Parameter(torch.randn(in_features, rank, dtype=dtype) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(rank, out_features, dtype=dtype))
    
    def forward(self, x):
        # x shape: (..., in_features)
        
        # Original linear transformation
        out = F.linear(x, self.weight, self.bias)
        
        # Add LoRA adaptation
        # (x @ A) @ B, scaled by alpha/rank
        lora_out = F.linear(x, self.lora_A.T, None)  # x @ A^T -> (..., rank)
        lora_out = F.linear(lora_out, self.lora_B.T, None)  # lora_out @ B^T -> (..., out)
        lora_out = lora_out * (self.lora_alpha / self.rank)
        
        return out + lora_out

# Test LoRA layer
layer = LoRALinear(768, 3072, rank=8)
layer = layer.to(device)

# Count parameters
total_params = sum(p.numel() for p in layer.parameters())
trainable_params = sum(p.numel() for p in layer.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")
print(f"\nPercentage trainable: {trainable_params / total_params * 100:.3f}%")

# Forward pass
x = torch.randn(2, 10, 768, device=device)  # batch=2, seq_len=10
output = layer(x)
print(f"\nInput shape: {x.shape}")
print(f"Output shape: {output.shape}")

# Gradient check
loss = output.sum()
loss.backward()
print(f"\nGradient on lora_A: {layer.lora_A.grad is not None}")
print(f"Gradient on lora_B: {layer.lora_B.grad is not None}")
print(f"Gradient on original weight: {layer.weight.grad is None}")

## Real-World Example 1: LoRA Fine-tuning with HuggingFace + PEFT

Use the PEFT library for practical LoRA fine-tuning

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Note: This requires PEFT library
# For demo, we'll show the concept

try:
    from peft import get_peft_model, LoraConfig, TaskType
    
    # Load model
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=2
    )
    
    # Configure LoRA
    lora_config = LoraConfig(
        r=8,                                  # Rank
        lora_alpha=16,                        # Scaling
        target_modules=["query", "value"],   # Which layers to apply LoRA
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.SEQ_CLS
    )
    
    # Wrap model with LoRA
    model = get_peft_model(model, lora_config)
    
    # Print trainable params
    model.print_trainable_parameters()
    
    print("\nLoRA successfully applied!")
    
except ImportError:
    print("PEFT library not installed. Showing conceptual example instead.")
    print("\nTo use LoRA in practice:")
    print("  pip install peft")
    print("\nThen:")
    print("  from peft import get_peft_model, LoraConfig")
    print("  lora_config = LoraConfig(r=8, lora_alpha=16, ...)")
    print("  model = get_peft_model(model, lora_config)")

## Real-World Example 2: Memory and Speed Comparison

Compare LoRA vs. full fine-tuning

In [ ]:
# Simulated benchmarks
model_sizes = ['7B', '13B', '70B', '175B']
model_params = [7, 13, 70, 175]  # Billions

# Memory usage for training (in GB)
# Full fine-tune: need gradient buffer (4x model size approximately)
full_ft_memory = np.array(model_params) * 4

# LoRA (r=8): only LoRA gradients, much smaller
# Rough estimate: 5% of full fine-tune memory
lora_memory = full_ft_memory * 0.05

# Training time (relative to 7B full fine-tune = 8 hours)
full_ft_time = np.array([1.0, 2.0, 4.0, 10.0])  # Relative multiplier
lora_time = np.array([0.1, 0.2, 0.5, 1.2])      # Much faster

# Plotting
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Memory comparison
x = np.arange(len(model_sizes))
width = 0.35

axes[0].bar(x - width/2, full_ft_memory, width, label='Full Fine-Tune', alpha=0.8)
axes[0].bar(x + width/2, lora_memory, width, label='LoRA (r=8)', alpha=0.8)
axes[0].set_ylabel('GPU Memory (GB)')
axes[0].set_title('Memory Usage Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_sizes)
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(alpha=0.3)

# Time comparison
axes[1].bar(x - width/2, full_ft_time, width, label='Full Fine-Tune', alpha=0.8)
axes[1].bar(x + width/2, lora_time, width, label='LoRA (r=8)', alpha=0.8)
axes[1].set_ylabel('Training Time (relative to 7B full)')
axes[1].set_title('Training Speed Comparison')
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_sizes)
axes[1].legend()
axes[1].set_yscale('log')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Key insights:")
for i, size in enumerate(model_sizes):
    mem_reduction = full_ft_memory[i] / lora_memory[i]
    time_reduction = full_ft_time[i] / lora_time[i]
    print(f"\n{size} model:")
    print(f"  Memory: {mem_reduction:.0f}x smaller with LoRA")
    print(f"  Speed: {time_reduction:.0f}x faster with LoRA")

## Real-World Example 3: LoRA Rank Selection

How to choose the right rank for your task

In [ ]:
# Simulated effect of rank on accuracy and efficiency
ranks = [2, 4, 8, 16, 32, 64]

# Accuracy (asymptotic approach to full fine-tune accuracy)
full_ft_accuracy = 0.92
accuracies = np.array([0.875, 0.900, 0.915, 0.920, 0.921, 0.922])  # Diminishing returns

# Parameters (relative to rank=8)
params_relative = np.array(ranks) / 8

# Efficiency: accuracy per parameter
efficiency = accuracies / params_relative

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Accuracy vs rank
axes[0].plot(ranks, accuracies, 'o-', linewidth=2, markersize=8, label='LoRA')
axes[0].axhline(full_ft_accuracy, color='r', linestyle='--', label='Full Fine-Tune')
axes[0].set_xlabel('Rank')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Accuracy vs. LoRA Rank')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Parameters vs rank
axes[1].plot(ranks, params_relative, 's-', linewidth=2, markersize=8)
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Relative Parameters')
axes[1].set_title('Model Size vs. Rank')
axes[1].grid(alpha=0.3)

# Efficiency (accuracy per parameter)
axes[2].plot(ranks, efficiency, '^-', linewidth=2, markersize=8, color='green')
axes[2].axvline(8, color='r', linestyle='--', alpha=0.5, label='Recommended')
axes[2].set_xlabel('Rank')
axes[2].set_ylabel('Accuracy per Parameter')
axes[2].set_title('Efficiency: Accuracy per Param')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Rank Selection Guide:")
print("========================")
print("r=2-4:   Tiny models, extreme constraints")
print("r=8:     DEFAULT - good for most tasks")
print("r=16:    Better accuracy if you have extra compute")
print("r=32:    Large models, complex tasks")
print("r>32:    Rarely needed, diminishing returns")

## Key Takeaways

**LoRA Concept:**
- Don't update full weight matrix W
- Update low-rank W_delta = B @ A (B: d×r, A: r×d, r << d)
- Frozen W reduces memory, only train LoRA matrices

**Why Low-Rank Works:**
- Task-specific adaptations don't need full rank
- Pre-trained models already capture general patterns
- Few directions matter for each task (r=8-16 enough)
- Empirically: 99% of full fine-tune quality with 1% of params

**Practical Benefits:**
- Memory: 100x smaller (fits on consumer GPUs)
- Speed: 10x faster training
- Artifacts: Save LoRA only (1-10MB per task)
- Flexibility: Swap adapters, train multiple tasks

**Rank Selection:**
- Default: r=8 (good across tasks)
- Increase if underfitting
- Diminishing returns after r=32

**LoRA in Production:**
- Train LoRA on GPU
- Merge LoRA into model for single-file deployment
- Or keep separate and swap at inference
- Works with quantization (QLoRA)